## Import necessary libraries

In [2]:
# Import necessary libraries
import numpy as np
import open3d as o3d
import laspy
import os
import pandas as pd
from scipy.spatial import Delaunay
import trimesh
from pygltflib import GLTF2, BufferFormat
import matplotlib.pyplot as plt

## Defining the functions for mesh reconstruction and simplfication

In [3]:
def load_ply_point_cloud(ply_file_path):
    point_cloud = o3d.io.read_point_cloud(ply_file_path)
    
    if point_cloud.has_colors():
        print("RGB values found in the .ply file.")
    else:
        print("No RGB values found in the .ply file.")
    return point_cloud

def load_laz_point_cloud(laz_file_path, decimation_factor=10):
    """
    Load a .laz point cloud file into an Open3D point cloud with optional decimation.
    
    Args:
    - laz_file_path: The path to the .laz file.
    - decimation_factor: The factor by which to downsample the point cloud. Default is 1 (no downsampling).
    
    Returns:
    - point_cloud: The loaded Open3D PointCloud object.
    """
    # Open the .laz file using laspy
    with laspy.open(laz_file_path) as las_file:
        las_data = las_file.read()

    # Extract the X, Y, Z coordinates
    xyz = np.vstack((las_data.x, las_data.y, las_data.z)).transpose()

    # Apply decimation by selecting every nth point based on the decimation factor
    if decimation_factor > 1:
        print(f"Original point cloud has {len(xyz)} points.")
        xyz = xyz[::decimation_factor]
        print(f"Decimated point cloud to {len(xyz)} points.")

    # Create the Open3D point cloud object
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(xyz)

    # Check for RGB color information and apply decimation if available
    if hasattr(las_data, 'red') and hasattr(las_data, 'green') and hasattr(las_data, 'blue'):
        red = las_data.red[::decimation_factor] / 65535.0
        green = las_data.green[::decimation_factor] / 65535.0
        blue = las_data.blue[::decimation_factor] / 65535.0
        rgb = np.vstack((red, green, blue)).transpose()
        point_cloud.colors = o3d.utility.Vector3dVector(rgb)
    else:
        print("No RGB color information found in .laz file.")

    return point_cloud


def load_point_cloud_from_dataframe(pcd_df):
    """
    Load a point cloud from a Pandas DataFrame and convert it to an Open3D point cloud.
    
    Args:
    - pcd_df (pd.DataFrame): DataFrame with columns 'X', 'Y', 'Z' for coordinates, and optionally 'R', 'G', 'B' for color.
    
    Returns:
    - o3d.geometry.PointCloud: The loaded point cloud in Open3D format.
    """
    # Ensure the data is in the correct format (float)
    points = np.array(pcd_df[['X', 'Y', 'Z']], dtype=np.float64)

    # Create the point cloud object
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(points)
    
    # Check if color information exists in the DataFrame and add it if available
    if all(col in pcd_df.columns for col in ['R', 'G', 'B']):
        colors = np.array(pcd_df[['R', 'G', 'B']], dtype=np.float64) / 255.0  # Normalize colors
        point_cloud.colors = o3d.utility.Vector3dVector(colors)
    else:
        print("No RGB color information found in DataFrame.")
    
    return point_cloud

def load_xyz_point_cloud(xyz_file_path):
    """
    Load an .xyz point cloud file into a Pandas DataFrame and convert it to an Open3D point cloud.

    Args:
    - xyz_file_path: The path to the .xyz file

    Returns:
    - o3d.geometry.PointCloud: The loaded point cloud in Open3D format.
    """
    try:
        # Load the file using ';' as the delimiter
        pcd_df = pd.read_csv(xyz_file_path, sep=";", header=0)
        
        # Select only the relevant columns (X, Y, Z, and optionally R, G, B)
        if {'X', 'Y', 'Z'}.issubset(pcd_df.columns):
            if {'R', 'G', 'B'}.issubset(pcd_df.columns):
                # If RGB is available, pass the full XYZRGB DataFrame
                pcd_df = pcd_df[['X', 'Y', 'Z', 'R', 'G', 'B']]
            else:
                # If RGB is not available, use only XYZ columns
                pcd_df = pcd_df[['X', 'Y', 'Z']]
            
            # Convert the DataFrame to a point cloud
            return load_point_cloud_from_dataframe(pcd_df)
        else:
            raise ValueError("The file does not contain the required X, Y, Z columns.")
    
    except Exception as e:
        print(f"Error loading .xyz point cloud: {e}")
        return None

def load_point_cloud(file_path, decimator_factor=50):
    file_extension = os.path.splitext(file_path)[-1].lower()
    if file_extension == ".laz" or file_extension == ".las":
        print("Loading .laz point cloud...")
        return load_laz_point_cloud(file_path, decimation_factor=decimator_factor)
    elif file_extension == ".xyz":
        print("Loading .xyz point cloud...")
        return load_xyz_point_cloud(file_path)
    elif file_extension == ".ply":
        print("Loading .ply point cloud...")
        return load_ply_point_cloud(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_extension}")


# New function to perform ball pivoting
def perform_ball_pivoting(point_cloud):
    """
    Estimate normals, calculate radius based on nearest neighbor distances,
    and perform the ball-pivoting algorithm to create a mesh.
    """
    # Step 4: Estimate normals using KD-Tree for efficiency
    if not point_cloud.has_normals():
        print("Estimating normals using KD-Tree...")
        point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    
    # Orient the normals consistently with the tangent plane to avoid issues in meshing
    point_cloud.orient_normals_consistent_tangent_plane(10)
    
    # Step 5: Estimate nearest neighbor distance and calculate radius for ball-pivoting
    distances = point_cloud.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radius = 3.0 * avg_dist  # Increase radius for faster mesh generation
    print(f"Average neighbor distance = {avg_dist:.6f}")
    
    # Step 6: Create a mesh using the Ball-Pivoting algorithm with the calculated radius
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        point_cloud,
        o3d.utility.DoubleVector([radius, radius * 2])
    )
    double_sided_mesh = create_double_sided_mesh(mesh)
    
    # Step 7: Visualize the mesh with the original point cloud
    print("Mesh generated using ball-pivoting.")
    # o3d.visualization.draw_plotly([point_cloud, double_sided_mesh], window_name='Ball-Pivoting Mesh', width=1200, height=800)

    return  double_sided_mesh

# Step 8: Mesh simplification using Quadric Decimation
def quadratic_decimation_simplify_mesh(mesh, target_triangle_count=10000):
    print(f"Original mesh has {len(mesh.triangles)} triangles.")
    simplified_mesh = mesh.simplify_quadric_decimation(target_triangle_count)
    print(f"Simplified mesh to {len(simplified_mesh.triangles)} triangles.")
    return simplified_mesh

def clustering_simplify_with_average(mesh, voxel_size=0.05):
    """
    Simplify a mesh using vertex clustering with average contraction.

    Args:
    - mesh (o3d.geometry.TriangleMesh): The input mesh to be simplified.
    - voxel_size (float): The size of the voxel grid for clustering vertices.

    Returns:
    - simplified_mesh (o3d.geometry.TriangleMesh): The simplified mesh.
    """
    # Apply vertex clustering simplification with average contraction
    simplified_mesh = mesh.simplify_vertex_clustering(
        voxel_size=voxel_size,
        contraction=o3d.geometry.SimplificationContraction.Average)
    
    return simplified_mesh

def create_double_sided_mesh(mesh):
    """
    Create a double-sided mesh by inverting the faces to create back-facing triangles.
    
    Args:
    - mesh (o3d.geometry.TriangleMesh): The input mesh to be converted to double-sided.
    
    Returns:
    - double_sided_mesh (o3d.geometry.TriangleMesh): The resulting double-sided mesh.
    """
    # Original vertices and faces
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.triangles)

    # Invert the faces to create back-facing triangles
    inverted_faces = faces[:, ::-1]

    # Create a new Open3D mesh with the same vertices and inverted faces
    double_sided_mesh = o3d.geometry.TriangleMesh()
    double_sided_mesh.vertices = o3d.utility.Vector3dVector(vertices)
    double_sided_mesh.triangles = o3d.utility.Vector3iVector(inverted_faces)

    # Assign vertex colors if available
    if mesh.has_vertex_colors():
        colors = np.asarray(mesh.vertex_colors)
        double_sided_mesh.vertex_colors = o3d.utility.Vector3dVector(colors)

    # Compute vertex normals for visualization
    double_sided_mesh.compute_vertex_normals()
    
    return double_sided_mesh


def convert_open3d_to_trimesh(double_sided_mesh):
    """
    Convert an Open3D mesh to a Trimesh object and set up a double-sided material.

    Args:
    - double_sided_mesh (o3d.geometry.TriangleMesh): The input Open3D mesh to be converted.

    Returns:
    - trimesh_mesh (trimesh.Trimesh): The resulting Trimesh object with double-sided material.
    """
    # Convert Open3D mesh to Trimesh
    vertices = np.asarray(double_sided_mesh.vertices)
    faces = np.asarray(double_sided_mesh.triangles)
    trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces)

    # Set up a double-sided material in Trimesh
    material = trimesh.visual.material.SimpleMaterial()
    material.doubleSided = True
    trimesh_mesh.visual.material = material

    return trimesh_mesh

## Loading the .las file (Point Cloud) and decimating it

In [31]:

# Path to point cloud file (either .laz, .xyz, or .ply)
input_file_path = "../DATA/"
file_name = "hoornestr.las"
point_cloud_file = input_file_path + file_name

# Load point cloud based on file type
point_cloud = load_point_cloud(point_cloud_file, decimator_factor=10)
print(f"Loaded point cloud has {len(point_cloud.points)} points.")

Loading .laz point cloud...
Original point cloud has 51379614 points.
Decimated point cloud to 5137962 points.
Loaded point cloud has 5137962 points.


In [33]:
# Visualize the point cloud with Open3D 
point_cloud_center = point_cloud.get_center()
point_cloud.translate(-point_cloud_center)
o3d.visualization.draw_geometries([point_cloud], window_name='Point Cloud', width=1200, height=800)

In [36]:
retained_ratio = 0.2
sampled_point_cloud = point_cloud.random_down_sample(retained_ratio)
o3d.visualization.draw_geometries([sampled_point_cloud], window_name = "Random Sampling")

In [37]:
#%% 3.2. Statistical outlier filter 
nn = 16
std_multiplier = 10

#The statistical outlier removal filter returns the point cloud and the point indexes
filtered_pcd, filtered_idx = point_cloud.remove_statistical_outlier(nn, std_multiplier)

#Visualizing the points filtered
outliers = point_cloud.select_by_index(filtered_idx, invert=True)
outliers.paint_uniform_color([1, 0, 0])

o3d.visualization.draw_geometries([filtered_pcd, outliers])

## Downsampling the Point Cloud

In [44]:
# Voxel downsampling
voxel_size = 0.2
down_sampled_point_cloud = filtered_pcd.voxel_down_sample(voxel_size)
print(f"Voxel downsampled point cloud has {len(point_cloud.points)} points.")
o3d.visualization.draw_geometries([down_sampled_point_cloud], window_name='Voxel Downsampled Point Cloud', width=1200, height=800)


Voxel downsampled point cloud has 227936 points.


In [46]:
## Estimate normals
nn_distance = np.mean(point_cloud.compute_nearest_neighbor_distance())
print(nn_distance)
#setting the radius search to compute normals
radius_normals=nn_distance*4

down_sampled_point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normals, max_nn=16), fast_normal_computation=True)
o3d.visualization.draw_geometries([down_sampled_point_cloud], window_name='Normals', width=1200, height=800)

0.06761977032528985


## Ball Pivoting Mesh reconstruction to get the mesh from the point cloud

In [47]:
# Perform ball-pivoting mesh reconstruction
ball_pivoting_mesh = perform_ball_pivoting(down_sampled_point_cloud)

Average neighbor distance = 0.129457
Mesh generated using ball-pivoting.


In [48]:
# Visualize the mesh with the original point cloud
o3d.visualization.draw_geometries([ball_pivoting_mesh], window_name='Ball-Pivoting Mesh', width=1200, height=800)

## Simplify Mesh using Quadratic Decimation

In [49]:
# Simplify the mesh using Quadric Decimation
target_triangle_count = len(ball_pivoting_mesh.triangles) // 10 # Target is 10% of original triangles
simplified_mesh = quadratic_decimation_simplify_mesh(ball_pivoting_mesh, target_triangle_count)

Original mesh has 89471 triangles.
Simplified mesh to 8947 triangles.


In [50]:
# Visualize the simplified mesh
o3d.visualization.draw_geometries_with_editing([simplified_mesh], window_name='Simplified Mesh', width=1200, height=800)

## Simplify Mesh with Vertex Clustering

In [51]:
# Simplify the mesh using vertex clustering with average contraction
simplified_mesh = clustering_simplify_with_average(ball_pivoting_mesh, voxel_size=0.5)
o3d.visualization.draw_geometries([simplified_mesh], window_name='Vertex Clustering Mesh', width=1200, height=800)

## Create a mesh from Alpha Shape and use alpha to reduce the detail

In [60]:
# Point cloud to mesh with compute alpha shape
alpha = 0.77 # Alpha value for alpha shape: smaller values result in more detailed mesh
simplified_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(down_sampled_point_cloud, alpha)
simplified_mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([simplified_mesh], window_name='Alpha Shape Mesh', width=1200, height=800)

## Export Mesh as a GLB file with Trimesh

In [61]:
# Convert Open3D mesh to Trimesh and set up double-sided material
trimesh_mesh = convert_open3d_to_trimesh(simplified_mesh)

# Step 2: Export to glTF using Trimesh (as intermediate format)
output_glb_path = f"../DATA/RESULTS/{file_name.split('.')[0]}_mesh.glb"
trimesh_mesh.export(output_glb_path, file_type='glb')

b'glTF\x02\x00\x00\x00\x84\xcc\x1b\x00\xa4\x03\x00\x00JSON{"scene":0,"scenes":[{"nodes":[0]}],"asset":{"version":"2.0","generator":"https://github.com/mikedh/trimesh"},"accessors":[{"componentType":5125,"type":"SCALAR","bufferView":0,"count":317010,"max":[46068],"min":[0]},{"componentType":5126,"type":"VEC3","byteOffset":0,"bufferView":1,"count":46069,"max":[15.018691062927246,22.595064163208008,4.657877445220947],"min":[-15.504380226135254,-20.72886848449707,-3.246675968170166]}],"meshes":[{"name":"geometry_0","extras":{"processed":true},"primitives":[{"attributes":{"POSITION":1},"indices":0,"mode":4,"material":0}]}],"materials":[{"pbrMetallicRoughness":{"baseColorFactor":[0.4,0.4,0.4,1.0],"roughnessFactor":0.9036020036098448},"doubleSided":false}],"nodes":[{"name":"world","children":[1]},{"name":"geometry_0","mesh":0}],"buffers":[{"byteLength":1820868}],"bufferViews":[{"buffer":0,"byteOffset":0,"byteLength":1268040},{"buffer":0,"byteOffset":1268040,"byteLength":552828}]}   \xc4\xc8\x

## RANSAC Multi-Plane Segmentation with Euclidean Consideration

In [67]:
pcd = down_sampled_point_cloud
rest = pcd

pt_to_plane_dist = nn_distance + np.std(pcd.compute_nearest_neighbor_distance())
# pt_to_plane_dist = 0.5

# Segment the point cloud into multiple planes
segment_models = {}
segments = {}
max_plane_idx = 6



for i in range(max_plane_idx):
    colors = plt.get_cmap("tab10")(i)
    segment_models[i], inliers = rest.segment_plane(distance_threshold=float(pt_to_plane_dist), ransac_n=3, num_iterations=1000)
    segments[i] = rest.select_by_index(inliers)
    
    # labels = np.array(segments[i].cluster_dbscan(eps=0.01, min_points=int(len(inliers)/10)))
    labels = np.array(segments[i].cluster_dbscan(eps=pt_to_plane_dist * 3, min_points=10))
    candidates = [len(np.where(labels == j)[0]) for j in np.unique(labels)]
    best_candidate = int(np.unique(labels)[np.where(candidates == np.max(candidates))[0][0]])
    print("the best candidate is: ", best_candidate)
    rest = rest.select_by_index(inliers, invert=True) + segments[i].select_by_index(list(np.where(labels != best_candidate)[0]))
    segments[i] = segments[i].select_by_index(list(np.where(labels == best_candidate)[0]))
    segments[i].paint_uniform_color(list(colors[:3]))
    print("pass", i + 1, "/", max_plane_idx, "done.")


labels = np.array(rest.cluster_dbscan(eps=0.05, min_points=5))
max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")

colors = plt.get_cmap("tab10")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
rest.colors = o3d.utility.Vector3dVector(colors[:, :3])

# Visualize the segmented point cloud
o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)]+[rest])
o3d.visualization.draw_geometries([segments[i] for i in range(max_plane_idx)], window_name='Segmented Planes', width=1200, height=800)


the best candidate is:  0
pass 1 / 6 done.
the best candidate is:  -1
pass 2 / 6 done.
the best candidate is:  1
pass 3 / 6 done.
the best candidate is:  0
pass 4 / 6 done.
the best candidate is:  3
pass 5 / 6 done.
the best candidate is:  -1
pass 6 / 6 done.
point cloud has 0 clusters


In [68]:
# Create a mesh of the segmented planes
alpha = 1.0
plane_meshes = []
for i in range(max_plane_idx):
    plane_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(segments[i], alpha)
    plane_mesh.compute_vertex_normals()
    plane_meshes.append(plane_mesh)

# Visualize the segmented planes as meshes
o3d.visualization.draw_geometries(plane_meshes, window_name='Segmented Planes Mesh', width=1200, height=800)